# Lab 5：用于 RAG 的 Agentic Document Extraction

在本实验中，你将使用 ADE 构建一个检索增强生成（RAG）流水线。RAG 通过检索相关文档块并基于这些上下文生成答案，使 LLM 能够回答关于文档的问题。

**学习目标：**
- 理解 RAG 的三个阶段：预处理、检索和生成
- 使用 OpenAI embeddings 搭建本地 ChromaDB 向量数据库
- 在查询中结合视觉溯源，实现可追踪的信息检索


## 背景

当问题与文档中的原始措辞不完全一致时，传统关键词搜索会失效。RAG 将语义搜索分为三部分：
1. 预处理：将文档解析为块，嵌入为向量，并存入数据库
2. 检索：将查询转换为向量，找到语义上相近的块
3. 生成：将检索到的块作为上下文传给 LLM，以生成有依据的答案

你将使用 Apple 的 10-K SEC 申报文件（74 页），它已通过 ADE 预先解析（共 453 个块）。每个块都包含文本、边界框坐标、页码和块类型（例如 text、table、figure）。


## 大纲

- [1. 导入库](#1)
- [2. 加载 ADE 解析数据](#2)
- [3. 设置向量数据库](#3)
- [4. 查询数据库](#4)
- [5. 混合搜索](#5)
- [6. 使用 LangChain 构建 RAG](#6)


<a id="1"></a>

## 1. 导入库

加载以下库：

- **openai**：生成向量嵌入（`text-embedding-3-small`）
- **chromadb**：用于存储和查询向量嵌入的向量数据库
- **langchain**：构建 RAG 链的框架

另外，我们还提供了一个脚本 `helper.py`，其中包含一些用于裁剪图像和叠加边界框的实用函数。  


In [ ]:
import os
import json
import re
from pathlib import Path
from dotenv import load_dotenv

from IPython.display import display, Image, IFrame, Markdown, JSON 

import helper

# OpenAI & ChromaDB - Embedding + Vector Store
import openai
import chromadb

# Langchain 
from langchain.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

# Load environment variables from .env
_ = load_dotenv(override=True)

<a id="2"></a>

## 2. 加载 ADE 解析数据

Apple 10-K 文档已经使用 ADE 预先解析。输出包括：
- Markdown：转换后的完整结构化 markdown，带有锚点标签
- JSON：单独的内容块及其元数据（边界框、页码、块类型）

这些预计算输出使你可以专注于 RAG 流水线本身。在生产环境中，你通常会直接调用 ADE Parse API。


In [ ]:
DOC_PATH = Path("apple_10k.pdf")
helper.print_document(DOC_PATH)

设置预计算 ADE 输出的路径：


In [ ]:
# Setting Directories and Paths
OUTPUT_DIR = Path("./ade_outputs")
ADE_JSON_PATH = OUTPUT_DIR / "apple_10k_chunks.json"  
ADE_MD_PATH = OUTPUT_DIR / "apple_10k.md"

预览解析后的 markdown。注意其中的锚点 ID，它们可以把提取出的信息链接回源位置。


In [ ]:
# Load and display markdown preview
print("\n Parsed Output (Page 1):")
with open(ADE_MD_PATH, "r", encoding="utf-8") as f:
    markdown_content = f.read()
    # Find first page content (up to first page break or 500 chars)
    first_page = markdown_content[:500]
    print(first_page + "...")

加载 JSON 格式的文档块。每个块包含：
- `chunk_id`：唯一标识符（与 markdown 中的锚点 ID 对应）
- `chunk_type`：`"text"`、`"table"` 或 `"figure"`
- `text`：实际内容
- `bbox`：边界框坐标 `[x0, y0, x1, y1]`（归一化到 0-1）
- `page`：页码（从 0 开始计数）


In [ ]:
# Load the existing JSON chunks
with open(ADE_JSON_PATH, "r", encoding="utf-8") as f:
    loaded_chunks = json.load(f)

print(f"Loaded {len(loaded_chunks)} saved chunks.")

# Show first chunk structure
print(f"\n Sample chunk structure:")
print(json.dumps(loaded_chunks[0], indent=2)[:400] + "...")

print("\n Ready to query!")

<a id="3"></a>

## 3. 设置向量数据库

设置 ChromaDB 来存储向量嵌入。ChromaDB 是一个轻量级向量数据库，支持：
- **HNSW indexing**：快速近似最近邻搜索
- **Metadata filtering**：按块类型、页码等进行查询过滤
- **Persistence**：内核重启后数据仍然保留

你将使用 OpenAI 的 `text-embedding-3-small` 将文本转换为 1536 维向量。


In [ ]:
# Setting Directory and Collectioon name
CHROMA_DB_PATH = Path("./chroma_db")
COLLECTION_NAME = "ade_documents"

# embeding model for vector database 
EMBEDDING_MODEL = "text-embedding-3-small"

# Instantiate the Chroma Client
chroma_client = chromadb.PersistentClient(path=CHROMA_DB_PATH)

# Create or Load ADE Collection
collection = chroma_client.get_or_create_collection(name=COLLECTION_NAME)

检查这些块是否已经在之前的运行中写入数据库：


In [ ]:
print(f"Checking for existing chunks in Chroma...")

# Get all existing chunk IDs from the collection
existing_result = collection.get(
                    ids=[chunk["chunk_id"] for chunk in loaded_chunks])
existing_ids = set(existing_result.get('ids', []))
print(f"Found {len(existing_ids)} existing chunks in collection")

对于每个新块，生成嵌入并连同元数据一起存储：
- `chunk_type`：按内容类型过滤（text 与 table）
- `page`：按页码过滤
- `bbox_*`：用于视觉溯源的边界框坐标


In [ ]:
print(f"Inserting new chunks into Chroma...")

added_count = 0
for i, chunk in enumerate(loaded_chunks):
    chunk_id = chunk["chunk_id"]
    
    # Add chunk if it does not exist
    if chunk_id not in existing_ids:
        text = chunk.get("text", "")
        
        # Skip empty chunks
        if not text or not text.strip():
            continue

        # Generate Embeddings for chunk text with OpenAI
        emb = openai.embeddings.create(
            input=text,
            model=EMBEDDING_MODEL
        ).data[0].embedding
        
        # Flatten Metadata (Simple Types Only)
        metadata = {
            "chunk_type": chunk.get("chunk_type", "unknown"),
            "page": chunk.get("page", 0)
        }
        
        # Add bbox coordinates to metadata
        bbox = chunk.get("bbox")
        if bbox and len(bbox) == 4:
            metadata["bbox_x0"] = float(bbox[0])
            metadata["bbox_y0"] = float(bbox[1])
            metadata["bbox_x1"] = float(bbox[2])
            metadata["bbox_y1"] = float(bbox[3])
        
        # Store in Chroma
        collection.add(
            documents=[text],
            ids=[chunk_id],
            metadatas=[metadata],
            embeddings=[emb]
        )
        
        added_count += 1
        
        # Progress indicator
        if (added_count) % 20 == 0:
            print(f"   Processed {added_count} new chunks...")

print(f"\n✓ Added {added_count} new chunks, skipped existing chunks")

<a id="4"></a>

## 4. 查询数据库

创建一个函数来查询向量数据库：
1. 嵌入查询：将问题转换为向量
2. 相似度搜索：找到与查询向量最接近的块
3. 按阈值过滤：只返回高于最小相似度的结果
4. 视觉溯源：显示源块的裁剪图像

相似度分数为 `1 - distance`，其中 distance 是向量之间的 L2（欧氏）距离。


In [ ]:
def rag_query(question, top_k=3, threshold=0.25, show_images=True):
    """
    Query the ADE Chroma index with a natural language question.
    Dynamically extracts and displays JUST the relevant chunk from 
    the PDF.
    
    Args:
        question (str): User query
        top_k (int): Max results to return
        threshold (float): Minimum similarity (1 - distance)
        show_images (bool): Display chunk-level grounding visualizations
    """
    # 1. Embed Query
    q_embed = openai.embeddings.create(
        model=EMBEDDING_MODEL,
        input=question
    ).data[0].embedding

    # 2. Query Chroma
    results = collection.query(
        query_embeddings=[q_embed],
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )

    print(f"\n Query: {question}\n")
    print("=" * 80)

    # 3. Parse Results
    retrieved_docs = results["documents"][0]
    retrieved_meta = results["metadatas"][0]
    retrieved_dists = results["distances"][0]
    retrieved_ids = results["ids"][0]

    found_any = False
    for i, (text, meta, dist, cid) in enumerate(zip(
        retrieved_docs, retrieved_meta, retrieved_dists, retrieved_ids
    )):
        similarity = 1 - dist
         # Skip Weak Matches
        if similarity >= threshold:
            
            found_any = True
            page_num = meta.get('page', 0)
            chunk_type = meta.get('chunk_type', 'unknown')
            
            print(f"\n  Result {i+1} (similarity={similarity:.3f}):")
            print(f"   Chunk ID: {cid}")
            print(f"   Type: {chunk_type}, Page: {page_num}" )
            print(f"   Text preview: {text[:200]}...")
            
            # Display chunk-level grounding image
            if show_images:
                # Extract bbox from metadata if available
                bbox = None
                if all(k in meta for k in ['bbox_x0', 'bbox_y0', 'bbox_x1', 'bbox_y1']):
                    bbox = [
                        meta['bbox_x0'],
                        meta['bbox_y0'],
                        meta['bbox_x1'],
                        meta['bbox_y1']
                    ]
                
                # Dynamically extract chunk image from PDF
                print(f"\n Dynamically extracting chunk from PDF...")
                chunk_img = helper.extract_chunk_image(
                    pdf_path=DOC_PATH,
                    page_num=page_num,
                    bbox=bbox,
                    highlight=True,
                    padding=10
                )
                
                if chunk_img:
                    print(f"{chunk_type.title()} chunk (cropped):")
                    display(Image(data=chunk_img))
                else:
                    print(f"Could not extract chunk image")
          
        
        print("-" * 80)

    if not found_any:
        print("No results above similarity threshold.")
    
    return results


print("RAG query function defined with dynamic chunk extraction")

测试这个查询函数。视觉溯源图像可帮助验证检索到的块是否包含相关信息：


In [ ]:
# Pass in Question in Natural Language, Top_K, and Threshold/L2 Distance
rag_query("What was Apple’s net sales in 2023?", 
          top_k=5, 
          threshold=0.32)

<a id="5"></a>

## 5. 混合搜索

语义搜索根据相似度检索文档块。混合搜索则通过 `where` 参数进一步加入元数据过滤。

例如，在询问收入数据时，只搜索表格，因为财务数据通常以表格形式呈现。这样可以避免检索到只是顺带提及收入的叙述性文本。


In [ ]:
q_embed = openai.embeddings.create(
    model=EMBEDDING_MODEL,
    input="What was Apple’s total revenue in 2023?",
).data[0].embedding

results = collection.query(
    query_embeddings=[q_embed],
    n_results=5,
    include=["documents", "metadatas", "distances"],
    where = {"chunk_type": "table"},
)

results["documents"]

<a id="6"></a>

## 6. RAG

使用 LangChain 将检索与生成结合起来。`create_retrieval_chain` 会构建一个流水线：
1. 接收用户问题
2. 从向量存储中检索相关块
3. 将它们格式化为 LLM 的上下文
4. 生成一个有依据的答案

这样就把手动嵌入和查询的步骤抽象成了一条可复用的链。


首先，用 LangChain 的 `Chroma` 类包装 ChromaDB collection，从而获得一个 retriever 接口：


In [ ]:
vectordb = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=OpenAIEmbeddings(model = EMBEDDING_MODEL),
    persist_directory=str(CHROMA_DB_PATH)
)

retriever = vectordb.as_retriever()

使用提示模板创建 RAG 链。`{context}` 占位符会被检索到的块填充：

<p style="background-color:#f7fff8; padding:15px; border-width:3px; border-color:#e0f0e0; border-style:solid; border-radius:6px"> 注意
&nbsp; <b>运行结果可能不同：</b> AI 聊天模型由于其非确定性特征，每次执行生成的输出都可能不同。如果你的结果与视频中展示的不同，请不要惊讶。</p>


In [ ]:
# Define prompt template
system_prompt = (
    "Use the following pieces of retrieved context to answer the "
    "user's question. "
    "If you don't know the answer, say that you don't know."
    "\n\n"
    "{context}"
)
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

# Initialize LLM
llm = ChatOpenAI(model="gpt-5-mini", temperature = 1)

# Create the RAG chain
rag_chain = create_retrieval_chain(retriever, prompt | llm)

# Invoke the chain (conceptual)
response = rag_chain.invoke({"input": 
                             "What were Apple net sales in 2023"})
print(response["answer"])


尝试一个需要理解跨年份趋势的问题：


In [ ]:
# Invoke the chain (conceptual)
response = rag_chain.invoke({
    "input": "How did total revenue trend between 2023 and 2022" 
                                       "for iPhone sales?"})
print(response["answer"])

## 总结

你已经基于 ADE 解析后的文档构建了一个完整的 RAG 流水线：

| 步骤 | 组件 | 作用 |
|------|-----------|--------|
| **Parse** | ADE Parse API | 将 PDF 转换为带有元数据的结构化块 |
| **Embed** | OpenAI `text-embedding-3-small` | 将文本转换为 1536 维向量 |
| **Store** | ChromaDB | 持久化保存带元数据的向量，以便快速检索 |
| **Query** | Similarity Search | 找到与查询语义相近的块 |
| **Retrieve** | Hybrid Search | 按块类型、页码等缩小结果范围 |
| **Generate** | LangChain RAG Chain | 基于检索到的上下文生成有依据的答案 |

此外，你还学会了通过视觉溯源来**验证**检索到的信息和生成的回答。

在下一课中，我们将学习如何在云端部署 ADE，通过事件驱动架构扩展我们的 RAG 应用，使其在有新文档出现时自动触发 ADE 进行处理。本实验帮助我们理解了驱动生产级流水线的核心逻辑。你在这里学到的 embeddings、相似度搜索、阈值调优和元数据过滤，都可以直接迁移到第 6 课。
